In [ ]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import pearsonr
import math
import pydicom
from show_keypoints import show_keypoints
import matplotlib.cm as cm
import scipy.stats as stats
from PIL import Image, ImageDraw
import os
import random
from bs4 import BeautifulSoup
import requests
from sklearn.model_selection import train_test_split

### Get software versions and device serial number

In [ ]:
path = "../UKB_xray_image_info/dcm/"

df = pd.DataFrame(columns=['file_name', 'software_version', 'device_serial_number', 'SERIAL_SOFTWARE'])

for f in os.listdir(path):
    if f.endswith(".dcm"):
        dcm = pydicom.dcmread(os.path.join(path, f))
        file_name = f.split(".dcm")[0]
        software_version = dcm.get((0x0018, 0x1020), 'Unknown').value
        device_serial_number = dcm.get((0x0018, 0x1000), 'Unknown').value
        SERIAL_SOFTWARE = "SS_" + str(device_serial_number) + "_" + str(software_version)

        # Append the data to the DataFrame
        df = df.append({
            'file_name': file_name,
            'software_version': software_version,
            'device_serial_number': device_serial_number,
            'SERIAL_SOFTWARE': SERIAL_SOFTWARE
        }, ignore_index=True)

# merge the serial_software with the EID
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm.csv')[['file_name', 'eid']]
dcm_info = hip_pheno.merge(df, on='file_name', how='inner')
dcm_info.to_csv("../UKB_xray_image_info/dicom_info.csv", index=False)

In [ ]:
dcm_info

### Make covar file for imaging individuals white british

In [ ]:
# get all fid and iid for white british
column_names = ['FID', 'IID', 'PID', 'MID', 'SEX', 'PHENOTYPE']
fam_df = pd.read_csv('./UKB_Imaging_Genetics/merged_sub_maf0.001_biallel_bbf_400k_white_british.fam', sep='\s+', header=None, names=column_names)

# rename PC columns and remove PC21-40
PC_df = pd.read_csv('./UKB_Imaging_Genetics/fid22009.csv', sep = ',')
PC_df.columns = ["eid"] + [f"PC{i}" for i in range(1, 41)]

columns_to_remove = [f"PC{i}" for i in range(21, 41)]  
PC_df = PC_df.drop(columns=columns_to_remove, errors='ignore')

covar_df = fam_df.merge(PC_df, left_on='IID', right_on='eid', how='inner')

# get age, sex, height information
fid_info = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20240104.csv", sep = ',')[['eid', 'age_imaging_visit', 'sex', 'standing_height_imaging_visit']]; fid_info
covar_df = covar_df.merge(fid_info, left_on='eid', right_on='eid', how='inner')

# remove redundant columns
covar_df.drop(['PID', 'MID', 'SEX', 'PHENOTYPE', 'eid'], axis=1, inplace=True)

# rename columns
covar_df.rename({'age_imaging_visit': 'AGE', 'sex': "SEX", 'standing_height_imaging_visit': 'HEIGHT'}, axis=1, inplace=True)
covar_df['HEIGHT2'] = covar_df['HEIGHT']**2

# add some covariates
covar_df['AGE2'] = covar_df['AGE']**2
covar_df['SEXAGE'] = covar_df['SEX'] * covar_df['AGE']
covar_df['SEXAGE2'] = covar_df['SEX'] * covar_df['AGE']**2

# add serial_software
dcm_info = pd.read_csv("../UKB_xray_image_info/dicom_info.csv")[['file_name', 'eid', 'SERIAL_SOFTWARE']]
dcm_info.sort_values(by = ['eid', 'file_name'], ascending=[True, True], inplace=True)
dcm_info.drop_duplicates(subset=['eid'], keep='first', inplace=True)
covar_df = covar_df.merge(dcm_info, left_on='IID', right_on='eid', how='inner')

# remove redundant columns
covar_df.drop(['eid'], axis=1, inplace=True)

# remove NA
covar_df.dropna(inplace=True)

# save
covar_df.to_csv('./UKB_Imaging_Genetics/GWAS_INPUT_DATA/covar_40k_imaging_20240104.txt', sep = " ", index=False)

In [ ]:
covar_df.columns

In [ ]:
covar_df.drop(columns=['HEIGHT2', 'AGE2', 'SEXAGE', 'SEXAGE2', 'file_name'], inplace=True)
covar_df.to_csv('./UKB_Imaging_Genetics/GWAS_INPUT_DATA/plink_covar_40k_imaging_20240104.txt', sep = " ", index=False)

In [ ]:
covar_df.columns

### Make covar file for 400k individuals white british

In [ ]:
# get all fid and iid for white british
column_names = ['FID', 'IID', 'PID', 'MID', 'SEX', 'PHENOTYPE']
fam_df = pd.read_csv('./UKB_Imaging_Genetics/merged_sub_maf0.001_biallel_bbf_400k_white_british.fam', sep='\s+', header=None, names=column_names)

# rename PC columns and remove PC21-40
PC_df = pd.read_csv('./UKB_Imaging_Genetics/fid22009.csv', sep = ',')
PC_df.columns = ["eid"] + [f"PC{i}" for i in range(1, 41)]

columns_to_remove = [f"PC{i}" for i in range(21, 41)]  
PC_df = PC_df.drop(columns=columns_to_remove, errors='ignore')

covar_df = fam_df.merge(PC_df, left_on='IID', right_on='eid', how='inner')

# get age, sex, height information
fid_info = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20240104.csv", sep = ',')[['eid', 'age', 'sex']]; fid_info
covar_df = covar_df.merge(fid_info, left_on='eid', right_on='eid', how='inner')

# remove redundant columns
covar_df.drop(['PID', 'MID', 'SEX', 'PHENOTYPE', 'eid'], axis=1, inplace=True)

# rename columns
covar_df.rename({'age': 'AGE', 'sex': "SEX"}, axis=1, inplace=True)

# add some covariates
covar_df['AGE2'] = covar_df['AGE']**2
covar_df['SEXAGE'] = covar_df['SEX'] * covar_df['AGE']
covar_df['SEXAGE2'] = covar_df['SEX'] * covar_df['AGE']**2

# remove NA
covar_df.dropna(inplace=True)

# save
covar_df.to_csv('./UKB_Imaging_Genetics/GWAS_INPUT_DATA/covar_400k_20240320.txt', sep = " ", index=False)

In [ ]:
covar_df

In [ ]:
covar_df = pd.read_csv("./UKB_Imaging_Genetics/GWAS_INPUT_DATA/covar_400k_20240320.txt", sep = " ")
plink_covar_df = covar_df.drop(columns=['AGE2', 'SEXAGE', 'SEXAGE2'])
plink_covar_df.to_csv('./UKB_Imaging_Genetics/GWAS_INPUT_DATA/plink_covar_400k_20240320.txt', sep = " ", index=False)

### Data cleaning

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm.csv'); hip_pheno

In [ ]:
covar_df = pd.read_csv('./UKB_Imaging_Genetics/GWAS_INPUT_DATA/covar_40k_imaging_20240104.txt', sep = " "); covar_df

In [ ]:
# keep white british
hip_pheno = hip_pheno[hip_pheno['file_name'].isin(covar_df['file_name'])]

In [ ]:
# filter ids that patients who don't want to share any more
remove_ids = pd.read_csv("../UKB_xray_image_info/remove_eids_20230504.csv", header=None)[0].tolist()
hip_pheno = hip_pheno[~hip_pheno['eid'].isin(remove_ids)]

In [ ]:
hip_pheno

In [ ]:
hip_pheno.to_csv('key_results/hip_select_pheno_cm_eid_flt.csv', index=False)

#### Z-score filter

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt.csv')

In [ ]:
hip_pheno

In [ ]:
# filter phenotypes

pheno = ['image_id', 'file_name', 'eid', 'sex', 'standing_height_imaging_visit', 'weight_imaging_visit', 'bmi_imaging_visit', 'age_imaging_visit',
         'pelvic_height', 'pelvic_width', 'pelvic_inlet_width', 'oblique_pelvic_inlet_length', 
         'subpubic_angle', 'bi_acetabular_width', 'iliac_isthmus_breadth', 'acetabular_diameter']
hip_pheno_flt = hip_pheno[pheno]; hip_pheno_flt

In [ ]:
hip_pheno_flt_eid_list = hip_pheno_flt[(np.abs(stats.zscore(hip_pheno_flt.iloc[:, 8:])) < 4).all(axis=1)]['eid']
len(hip_pheno_flt_eid_list)

In [ ]:
hip_pheno_flt = hip_pheno[hip_pheno['eid'].isin(hip_pheno_flt_eid_list)]; hip_pheno_flt

In [ ]:
hip_pheno_flt.to_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt.csv', index=False)

### Other skeletal traits

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt.csv')

In [ ]:
hip_pheno.columns

In [ ]:
# filter phenotypes

pheno = ['image_id', 'file_name', 'eid', 'sex', 'standing_height_imaging_visit', 'weight_imaging_visit', 'bmi_imaging_visit', 'age_imaging_visit',
         'head_diameter', 'trochanter_distance', 'shoulder_width', 'torso_length', 'humerus', 'femur', 'forearm', 'tibia']
hip_pheno_flt = hip_pheno[pheno]; hip_pheno_flt

In [ ]:
z_keep_eids = pd.read_csv("key_results/hip_select_pheno_cm_eid_flt_z_flt.csv")['eid'].tolist()

hip_pheno_flt = hip_pheno_flt[hip_pheno_flt['eid'].isin(z_keep_eids)]

hip_pheno_flt.to_csv("key_results/skeletal_cm_eid_flt_z_flt", index = False)

## Heritability

### Prepare data for h2g

#### use raw phenotype data (cm)

In [ ]:
hip_pheno_flt_eid_flt_cm = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt.csv'); hip_pheno_flt_eid_flt_cm

In [ ]:
# select the phenotypes for h2g
hip_pheno_gwas_cm = hip_pheno_flt_eid_flt_cm.iloc[:, 8:]

ids = hip_pheno_flt_eid_flt_cm['eid'].tolist()

hip_pheno_gwas_cm['FID'] = ids
hip_pheno_gwas_cm['IID'] = ids

# put FID and IID to the first two columns
cols = hip_pheno_gwas_cm.columns
cols = cols[-2:].tolist() + cols[:-2].tolist()
hip_pheno_gwas_cm = hip_pheno_gwas_cm[cols]

hip_pheno_gwas_cm

In [ ]:
hip_pheno_gwas_cm

In [ ]:
hip_pheno_flt_eid_flt_cm[['eid', 'eid']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_20240104.txt', index=False, header=False, sep=' ')

In [ ]:
hip_pheno_gwas_cm.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_cm_20240104.txt', index=False, sep = " ")
hip_pheno_gwas_cm.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_cm_no_header_20240104.txt', index=False, header=False, sep = " ")

#### Run gwas for height

In [ ]:
fid_info = pd.read_csv('../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20231028.csv'); fid_info

In [ ]:
both_height = fid_info[['eid', 'standing_height']].dropna()
male_height = fid_info[fid_info['sex'] == 1.0][['eid', 'standing_height']].dropna()
female_height = fid_info[fid_info['sex'] == 0.0][['eid', 'standing_height']].dropna()

both_height_eid = both_height['eid'].tolist()
male_height_eid = male_height['eid'].tolist()
female_height_eid = female_height['eid'].tolist()

gwas_standing_height_both = pd.DataFrame({"FID": both_height_eid, "IID": both_height_eid, "standing_height": both_height['standing_height'].tolist()})
gwas_standing_height_male = pd.DataFrame({"FID": male_height_eid, "IID": male_height_eid, "standing_height": male_height['standing_height'].tolist()})
gwas_standing_height_female = pd.DataFrame({"FID": female_height_eid, "IID": female_height_eid, "standing_height": female_height['standing_height'].tolist()})

In [ ]:
gwas_standing_height_both

In [ ]:
print(f'both_height_shape: {gwas_standing_height_both.shape}')
print(f'male_height_shape: {gwas_standing_height_male.shape}')
print(f'female_height_shape: {gwas_standing_height_female.shape}')

In [ ]:
gwas_standing_height_both.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_both_231107.txt', index=False, sep = " ")
gwas_standing_height_both.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_both_no_header_231107.txt', index=False, header=False, sep = " ")

gwas_standing_height_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_male_231107.txt', index=False, sep = " ")
gwas_standing_height_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_male_no_header_231107.txt', index=False, header=False, sep = " ")

gwas_standing_height_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_female_231107.txt', index=False, sep = " ")
gwas_standing_height_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_female_no_header_231107.txt', index=False, header=False, sep = " ")

In [ ]:
# removed imaged individuals

# get all fid and iid for imaged individuals
iid_img = pd.read_csv('key_results/hip_pheno_23_cm.csv')['eid'].tolist()
iid_400k = pd.read_csv("UKB_Imaging_Genetics/merged_sub_maf0.01_biallel_bbf_400k_white_british.fam", sep = " ", header=None)[0].tolist()

# both
gwas_standing_height_both = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_both_231107.txt', sep = " ")
gwas_standing_height_both = gwas_standing_height_both[gwas_standing_height_both['IID'].isin(iid_400k)]
gwas_standing_height_both_train = gwas_standing_height_both[~gwas_standing_height_both['IID'].isin(iid_img)]
gwas_standing_height_both_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_both_train_231128.txt', index=False, sep = " ")
gwas_standing_height_both_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_both_train_no_header_231128.txt', index=False, header=False, sep = " ")

# female
gwas_standing_height_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_female_231107.txt', sep = " ")
gwas_standing_height_female = gwas_standing_height_female[gwas_standing_height_female['IID'].isin(iid_400k)]
gwas_standing_height_female_train = gwas_standing_height_female[~gwas_standing_height_female['IID'].isin(iid_img)]
gwas_standing_height_female_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_female_train_231128.txt', index=False, sep = " ")
gwas_standing_height_female_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_female_train_no_header_231128.txt', index=False, header=False, sep = " ")

# male
gwas_standing_height_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_male_231107.txt', sep = " ")
gwas_standing_height_male = gwas_standing_height_male[gwas_standing_height_male['IID'].isin(iid_400k)]
gwas_standing_height_male_train = gwas_standing_height_male[~gwas_standing_height_male['IID'].isin(iid_img)]
gwas_standing_height_male_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_male_train_231128.txt', index=False, sep = " ")
gwas_standing_height_male_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/gwas_standing_height_male_train_no_header_231128.txt', index=False, header=False, sep = " ")

In [ ]:
print(f'both_train_shape: {gwas_standing_height_both_train.shape}')

print(f'female_train_shape: {gwas_standing_height_female_train.shape}')

print(f'male_train_shape: {gwas_standing_height_male_train.shape}')

#### add sex new phenotypes (head or shoulder / pelvic inlet size)

In [ ]:
hip_pheno_gwas_cm = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_cm_231109.txt', sep = " "); hip_pheno_gwas_cm

In [ ]:
# 2 for head or shoulder / oblique pelvic inlet width
hip_pheno_gwas_cm['head_divide_inlet_width'] = hip_pheno_gwas_cm['ear_left2ear_right'] / hip_pheno_gwas_cm['pelvic_inlet_width']
hip_pheno_gwas_cm['shoulder_divide_inlet_width'] = hip_pheno_gwas_cm['shoulder_width'] / hip_pheno_gwas_cm['pelvic_inlet_width']

# 2 for head or shoulder / oblique pelvic inlet length
hip_pheno_gwas_cm['head_divide_oblique_inlet_length'] = hip_pheno_gwas_cm['ear_left2ear_right'] / hip_pheno_gwas_cm['oblique_pelvic_inlet_length']
hip_pheno_gwas_cm['shoulder_divide_oblique_inlet_length'] = hip_pheno_gwas_cm['shoulder_width'] / hip_pheno_gwas_cm['oblique_pelvic_inlet_length']


# 2 for head or shoulder / pelvic inlet area
hip_pheno_gwas_cm['head_area_divide_pelvic_inlet_area'] = ((hip_pheno_gwas_cm['ear_left2ear_right'] / 2)**2 * math.pi) / hip_pheno_gwas_cm['pelvic_inlet_area']
hip_pheno_gwas_cm['shoulder_area_divide_pelvic_inlet_area'] = ((hip_pheno_gwas_cm['shoulder_width'] / 2)**2 * math.pi) / hip_pheno_gwas_cm['pelvic_inlet_area']

In [ ]:
hip_pheno

#### add 2 new phenotype as postive control: 1. arm:leg, 2. leg:torso

In [ ]:
hip_pheno_all = pd.read_csv('key_results/hip_pheno_23_cm.csv')

image_ids = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt.csv')['image_id'].tolist()

# filter individuals
hip_pheno_all = hip_pheno_all[hip_pheno_all['image_id'].isin(image_ids)]

# arm to torso ratio
hip_pheno_all['arm_divide_leg'] = ((hip_pheno_all['arm_123_left'] + hip_pheno_all['arm_123_right']) / 2) / ((hip_pheno_all['leg_123_left'] + hip_pheno_all['leg_123_right']) / 2)
hip_pheno_all['leg_divide_torso'] = ((hip_pheno_all['leg_123_left'] + hip_pheno_all['leg_123_right']) / 2) / hip_pheno_all['torso_length']
add_pheno = hip_pheno_all[['eid', 'arm_divide_leg', 'leg_divide_torso']]

# merge previous phenotypes
hip_pheno_gwas_cm = hip_pheno_gwas_cm.merge(add_pheno, left_on='FID', right_on='eid', how='left')
hip_pheno_gwas_cm.drop(columns=['eid'], inplace=True)

In [ ]:
hip_pheno_gwas_cm

In [ ]:
hip_pheno_gwas_cm.columns

#### filter phenotypes for gwas

In [ ]:
pheno = ['FID', 'IID', 'pelvic_height', 'pelvic_width', 'pelvic_inlet_width',
         'oblique_pelvic_inlet_length', 'subpubic_angle', 'bi_acetabular_width', 
         'iliac_isthmus_breadth', 'acetabular_diameter', 'pelvic_inlet_area',
         'leg_length']

hip_pheno_gwas_cm = hip_pheno_gwas_cm[pheno]

In [ ]:
hip_pheno_gwas_cm

In [ ]:
hip_pheno_gwas_cm.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_both_cm_20240104.txt', index=False, sep = " ")
hip_pheno_gwas_cm.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_both_cm_no_header_20240104.txt', index=False, header=False, sep = " ")

In [ ]:
hip_pheno_gwas_cm[['FID', 'IID']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_both_eids_20240104.txt', index=False, header=False, sep=' ')

### Divide to male and female 231106

In [ ]:
hip_pheno_gwas_cm = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_both_cm_231109.txt', sep = " "); hip_pheno_gwas_cm

In [ ]:
sex_df = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt.csv')[['eid', 'sex']]

In [ ]:
male_eids = sex_df[sex_df['sex'] == 1.0]['eid'].tolist()
female_eids = sex_df[sex_df['sex'] == 0.0]['eid'].tolist()
print(f"male_eids_length = {len(male_eids)}")
print(f"female_eids_length = {len(female_eids)}")

In [ ]:
hip_pheno_gwas_male = hip_pheno_gwas_cm[hip_pheno_gwas_cm['IID'].isin(male_eids)]
hip_pheno_gwas_female = hip_pheno_gwas_cm[hip_pheno_gwas_cm['IID'].isin(female_eids)]

print(f"male shape = {hip_pheno_gwas_male.shape}")
print(f"female shape = {hip_pheno_gwas_female.shape}")

hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_231109.txt', index=False, sep = " ")
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_no_header_231109.txt', index=False, header=False, sep = " ")


hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_231109.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_no_header_231109.txt', index=False, header=False, sep = " ")

### add birth weight/birth canal phenotypes

In [ ]:
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_231109.txt', sep = " ")
outcome_pheno = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20230916.csv")[['eid', 'birth_weight_first_child']]

hip_pheno_gwas_female = hip_pheno_gwas_female.merge(outcome_pheno, left_on="IID", right_on="eid")
hip_pheno_gwas_female.dropna(inplace=True)

hip_pheno_gwas_female['BW_devide_oblique_pelvic_inlet_length'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['oblique_pelvic_inlet_length']
hip_pheno_gwas_female['BW_devide_pelvic_inlet_width'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['pelvic_inlet_width']
hip_pheno_gwas_female['BW_devide_pelvic_inlet_area'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['pelvic_inlet_area']

hip_pheno_gwas_female = hip_pheno_gwas_female[['FID', 'IID', 'BW_devide_oblique_pelvic_inlet_length', 'BW_devide_pelvic_inlet_width', 'BW_devide_pelvic_inlet_area']]

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_devide_birth_canal_231228.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_devide_birth_canal_no_header_231228.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_female

### Divide to male and female 20240104

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt.csv'); hip_pheno

In [ ]:
# filter phenotypes

pheno = ['image_id', 'file_name', 'eid', 'sex', 'standing_height_imaging_visit', 'weight_imaging_visit', 'bmi_imaging_visit', 'age_imaging_visit',
         'pelvic_height', 'pelvic_width', 'pelvic_inlet_width', 'oblique_pelvic_inlet_length', 
         'subpubic_angle', 'bi_acetabular_width', 'iliac_isthmus_breadth', 'acetabular_diameter']
hip_pheno_flt = hip_pheno[pheno]

# male
hip_pheno_male_flt = hip_pheno_flt[hip_pheno_flt['sex'] == 1]
hip_pheno_male_flt_eid_list = hip_pheno_male_flt[(np.abs(stats.zscore(hip_pheno_male_flt.iloc[:, 8:])) < 4).all(axis=1)]['eid']

# z score filter
hip_pheno_male = hip_pheno[hip_pheno['eid'].isin(hip_pheno_male_flt_eid_list)]

# female
hip_pheno_female_flt = hip_pheno_flt[hip_pheno_flt['sex'] == 0]
hip_pheno_female_flt_eid_list = hip_pheno_female_flt[(np.abs(stats.zscore(hip_pheno_female_flt.iloc[:, 8:])) < 4).all(axis=1)]['eid']

# z score filter
hip_pheno_female = hip_pheno[hip_pheno['eid'].isin(hip_pheno_female_flt_eid_list)]

print(f"Male dataframe shape: {hip_pheno_male.shape}")
print(f"Female dataframe shape: {hip_pheno_female.shape}")

In [ ]:
hip_pheno_male.to_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt_male.csv', index=False)
hip_pheno_female.to_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt_female.csv', index=False)

In [ ]:
hip_pheno_male

In [ ]:
pheno = ['image_id', 'file_name', 'eid', 'sex', 'standing_height_imaging_visit', 'weight_imaging_visit', 'bmi_imaging_visit', 'age_imaging_visit',
         'pelvic_height', 'pelvic_width', 'pelvic_inlet_width', 'oblique_pelvic_inlet_length', 
         'subpubic_angle', 'bi_acetabular_width', 'iliac_isthmus_breadth', 'acetabular_diameter',
         'pelvic_inlet_area']

# male dataframe for gwas preparation
hip_pheno_gwas_male = hip_pheno_male[pheno]

hip_pheno_gwas_male = hip_pheno_male.iloc[:, 8:]

ids = hip_pheno_male['eid'].tolist()

hip_pheno_gwas_male['FID'] = ids
hip_pheno_gwas_male['IID'] = ids

# put FID and IID to the first two columns
cols = hip_pheno_gwas_male.columns
cols = cols[-2:].tolist() + cols[:-2].tolist()
hip_pheno_gwas_male = hip_pheno_gwas_male[cols]

# female dataframe for gwas preparation
hip_pheno_gwas_female = hip_pheno_female[pheno]

hip_pheno_gwas_female = hip_pheno_female.iloc[:, 8:]

ids = hip_pheno_female['eid'].tolist()

hip_pheno_gwas_female['FID'] = ids
hip_pheno_gwas_female['IID'] = ids

# put FID and IID to the first two columns
cols = hip_pheno_gwas_female.columns
cols = cols[-2:].tolist() + cols[:-2].tolist()
hip_pheno_gwas_female = hip_pheno_gwas_female[cols]

print(f"Male dataframe shape: {hip_pheno_gwas_male.shape}")
print(f"Female dataframe shape: {hip_pheno_gwas_female.shape}")

In [ ]:
hip_pheno_gwas_female

In [ ]:
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_20240104.txt', index=False, sep = " ")
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_no_header_20240104.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_20240104.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_no_header_20240104.txt', index=False, header=False, sep = " ")

In [ ]:
hip_pheno_gwas_male[['FID', 'IID']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_male_eids_20240104.txt', index=False, header=False, sep=' ')
hip_pheno_gwas_female[['FID', 'IID']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_female_eids_20240104.txt', index=False, header=False, sep=' ')

#### add birth weight/birth canal phenotypes

In [ ]:
outcome_pheno = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20240104.csv")[['eid', 'birth_weight_first_child']]

hip_pheno_gwas_female = hip_pheno_gwas_female.merge(outcome_pheno, left_on="IID", right_on="eid")
hip_pheno_gwas_female.dropna(inplace=True)

hip_pheno_gwas_female['BW_devide_oblique_pelvic_inlet_length'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['oblique_pelvic_inlet_length']
hip_pheno_gwas_female['BW_devide_pelvic_inlet_width'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['pelvic_inlet_width']
hip_pheno_gwas_female['BW_devide_pelvic_inlet_area'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['pelvic_inlet_area']

hip_pheno_gwas_female = hip_pheno_gwas_female[['FID', 'IID', 'BW_devide_oblique_pelvic_inlet_length', 'BW_devide_pelvic_inlet_width', 'BW_devide_pelvic_inlet_area']]

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_birth_canal_20240104.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_birth_canal_no_header_20240104.txt', index=False, header=False, sep = " ")

In [ ]:
pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_birth_canal_20240104.txt', sep = " ")

In [ ]:
hip_pheno_male[['eid', 'eid']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_male.txt', index=False, header=False, sep=' ')
hip_pheno_female[['eid', 'eid']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_female.txt', index=False, header=False, sep=' ')

#### add two new phenotypes (head or shoulder / pelvic inlet width)

In [ ]:
# male
hip_pheno_gwas_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_230713.txt', sep = " ")

hip_pheno_gwas_male['head_divide_inlet_width'] = hip_pheno_gwas_male['ear_left2ear_right'] / hip_pheno_gwas_male['pelvic_inlet_width']
hip_pheno_gwas_male['shoulder_divide_inlet_width'] = hip_pheno_gwas_male['shoulder_width'] / hip_pheno_gwas_male['pelvic_inlet_width']

hip_pheno_gwas_male_new = hip_pheno_gwas_male[['FID', 'IID', 'head_divide_inlet_width', 'shoulder_divide_inlet_width']]

hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_230829.txt', index=False, sep = " ")
hip_pheno_gwas_male_new.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_male_cm_230829.txt', index=False, sep = " ")

# female
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230713.txt', sep = " ")

hip_pheno_gwas_female['head_divide_inlet_width'] = hip_pheno_gwas_female['ear_left2ear_right'] / hip_pheno_gwas_female['pelvic_inlet_width']
hip_pheno_gwas_female['shoulder_divide_inlet_width'] = hip_pheno_gwas_female['shoulder_width'] / hip_pheno_gwas_female['pelvic_inlet_width']

hip_pheno_gwas_female_new = hip_pheno_gwas_female[['FID', 'IID', 'head_divide_inlet_width', 'shoulder_divide_inlet_width']]

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230829.txt', index=False, sep = " ")
hip_pheno_gwas_female_new.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_female_cm_230829.txt', index=False, sep = " ")

#### add five new phenotypes (2 for head or shoulder / oblique pelvic inlet length, 1 for pelvic inlet area, 2 for head area or shoulder area / pelvic inlet area)

In [ ]:
# male
hip_pheno_gwas_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_230829.txt', sep = " ")
# 2 for head or shoulder / oblique pelvic inlet length
hip_pheno_gwas_male['head_divide_oblique_inlet_length'] = hip_pheno_gwas_male['ear_left2ear_right'] / hip_pheno_gwas_male['oblique_pelvic_inlet_length']
hip_pheno_gwas_male['shoulder_divide_oblique_inlet_length'] = hip_pheno_gwas_male['shoulder_width'] / hip_pheno_gwas_male['oblique_pelvic_inlet_length']

# 1 for pelvic inlet area
hip_pheno_gwas_male['pelvic_inlet_area'] = (hip_pheno_gwas_male['pelvic_inlet_width'] / 2) * (hip_pheno_gwas_male['oblique_pelvic_inlet_length'] / 2) * math.pi

# 2 for head or shoulder / pelvic inlet area
hip_pheno_gwas_male['head_area_divide_pelvic_inlet_area'] = ((hip_pheno_gwas_male['ear_left2ear_right'] / 2)**2 * math.pi) / hip_pheno_gwas_male['pelvic_inlet_area']
hip_pheno_gwas_male['shoulder_area_divide_pelvic_inlet_area'] = ((hip_pheno_gwas_male['shoulder_width'] / 2)**2 * math.pi) / hip_pheno_gwas_male['pelvic_inlet_area']

hip_pheno_gwas_male_new = hip_pheno_gwas_male[['FID', 'IID', 
                                                'head_divide_oblique_inlet_length', 'shoulder_divide_oblique_inlet_length', 
                                                'pelvic_inlet_area',
                                                'head_area_divide_pelvic_inlet_area', 'shoulder_area_divide_pelvic_inlet_area']]

hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_230908.txt', index=False, sep = " ")
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_no_header_230908.txt', index=False, header=False, sep = " ")
hip_pheno_gwas_male_new.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_male_cm_230908.txt', index=False, sep = " ")

# female
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230829.txt', sep = " ")
# 2 for head or shoulder / oblique pelvic inlet length
hip_pheno_gwas_female['head_divide_oblique_inlet_length'] = hip_pheno_gwas_female['ear_left2ear_right'] / hip_pheno_gwas_female['oblique_pelvic_inlet_length']
hip_pheno_gwas_female['shoulder_divide_oblique_inlet_length'] = hip_pheno_gwas_female['shoulder_width'] / hip_pheno_gwas_female['oblique_pelvic_inlet_length']

# 1 for pelvic inlet area
hip_pheno_gwas_female['pelvic_inlet_area'] = (hip_pheno_gwas_female['pelvic_inlet_width'] / 2) * (hip_pheno_gwas_female['oblique_pelvic_inlet_length'] / 2) * math.pi

# 2 for head or shoulder / pelvic inlet area
hip_pheno_gwas_female['head_area_divide_pelvic_inlet_area'] = ((hip_pheno_gwas_female['ear_left2ear_right'] / 2)**2 * math.pi) / hip_pheno_gwas_female['pelvic_inlet_area']
hip_pheno_gwas_female['shoulder_area_divide_pelvic_inlet_area'] = ((hip_pheno_gwas_female['shoulder_width'] / 2)**2 * math.pi) / hip_pheno_gwas_female['pelvic_inlet_area']

hip_pheno_gwas_female_new = hip_pheno_gwas_female[['FID', 'IID', 
                                                'head_divide_oblique_inlet_length', 'shoulder_divide_oblique_inlet_length', 
                                                'pelvic_inlet_area',
                                                'head_area_divide_pelvic_inlet_area', 'shoulder_area_divide_pelvic_inlet_area']]

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230908.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_no_header_230908.txt', index=False, header=False, sep = " ")
hip_pheno_gwas_female_new.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_female_cm_230908.txt', index=False, sep = " ")

#### add 3 new phenotype for female: birth weight / birth canal phenos (3 phenotypes)

In [ ]:
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230908.txt', sep = " ")

outcome_pheno = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20230916.csv")[['eid', 'birth_weight_first_child']]

In [ ]:
hip_pheno_gwas_female = hip_pheno_gwas_female.merge(outcome_pheno, left_on="IID", right_on="eid")

In [ ]:
hip_pheno_gwas_female.dropna(inplace=True)

In [ ]:
hip_pheno_gwas_female['BW_devide_oblique_pelvic_inlet_length'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['oblique_pelvic_inlet_length']
hip_pheno_gwas_female['BW_devide_pelvic_inlet_width'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['pelvic_inlet_width']
hip_pheno_gwas_female['BW_devide_pelvic_inlet_area'] = \
    hip_pheno_gwas_female['birth_weight_first_child'] / hip_pheno_gwas_female['pelvic_inlet_area']

hip_pheno_gwas_female = hip_pheno_gwas_female[['FID', 'IID', 'BW_devide_oblique_pelvic_inlet_length', 'BW_devide_pelvic_inlet_width', 'BW_devide_pelvic_inlet_area']]

In [ ]:
hip_pheno_gwas_female

In [ ]:
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_devide_birth_canal_231005.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_BW_devide_birth_canal_no_header_231005.txt', index=False, header=False, sep = " ")

#### add 2 new phenotype as postive control: 1. arm:leg, 2. leg:torso

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_pheno_23_cm.csv')

image_ids = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt.csv')['image_id'].tolist()

# filter individuals
hip_pheno = hip_pheno[hip_pheno['image_id'].isin(image_ids)]

# new phenotypes
hip_pheno['arm_divide_leg'] = ((hip_pheno['arm_123_left'] + hip_pheno['arm_123_right']) / 2) / ((hip_pheno['leg_123_left'] + hip_pheno['leg_123_right']) / 2)
hip_pheno['leg_divide_torso'] = ((hip_pheno['leg_123_left'] + hip_pheno['leg_123_right']) / 2) / hip_pheno['torso_length']
add_pheno = hip_pheno[['eid', 'arm_divide_leg', 'leg_divide_torso']]

# process male

# merge previous phenotypes
hip_pheno_gwas_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_230908.txt', sep = " ")
hip_pheno_gwas_male = hip_pheno_gwas_male.merge(add_pheno, left_on='FID', right_on='eid', how='inner')
hip_pheno_gwas_male.drop(columns=['eid'], inplace=True)

hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_231011.txt', index=False, sep = " ")
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_no_header_231011.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_male[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_male_231011.txt', index=False, sep = " ")
hip_pheno_gwas_male[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_male_no_header_231011.txt', index=False, header=False, sep = " ")

# process female

# merge previous phenotypes
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230908.txt', sep = " ")
hip_pheno_gwas_female = hip_pheno_gwas_female.merge(add_pheno, left_on='FID', right_on='eid', how='inner')
hip_pheno_gwas_female.drop(columns=['eid'], inplace=True)

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_231011.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_no_header_231011.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_female[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_female_231011.txt', index=False, sep = " ")
hip_pheno_gwas_female[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_two_new_pheno_gwas_female_no_header_231011.txt', index=False, header=False, sep = " ")

In [ ]:
hip_pheno_gwas_male

In [ ]:
hip_pheno_gwas_female

#### add bi-acetabular width phentoype

In [ ]:
bi_acetabular_width = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt.csv')[['eid', 'bi_acetabular_width']]; bi_acetabular_width

In [ ]:
# process male

# merge previous phenotypes
hip_pheno_gwas_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_231011.txt', sep = " ")
hip_pheno_gwas_male = hip_pheno_gwas_male.merge(bi_acetabular_width, left_on='FID', right_on='eid', how='inner')
hip_pheno_gwas_male.drop(columns=['eid'], inplace=True)

hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_231025.txt', index=False, sep = " ")
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_no_header_231025.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_male[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso', 'bi_acetabular_width']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_three_new_pheno_gwas_male_231025.txt', index=False, sep = " ")
hip_pheno_gwas_male[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso', 'bi_acetabular_width']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_three_new_pheno_gwas_male_no_header_231025.txt', index=False, header=False, sep = " ")

# process female

# merge previous phenotypes
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_231011.txt', sep = " ")
hip_pheno_gwas_female = hip_pheno_gwas_female.merge(bi_acetabular_width, left_on='FID', right_on='eid', how='inner')
hip_pheno_gwas_female.drop(columns=['eid'], inplace=True)

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_231025.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_no_header_231025.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_female[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso', 'bi_acetabular_width']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_three_new_pheno_gwas_female_231025.txt', index=False, sep = " ")
hip_pheno_gwas_female[['FID', 'IID', 'arm_divide_leg', 'leg_divide_torso', 'bi_acetabular_width']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_three_new_pheno_gwas_female_no_header_231025.txt', index=False, header=False, sep = " ")

In [ ]:
hip_pheno_gwas_male.shape

In [ ]:
hip_pheno_gwas_female.shape

#### add leg length phentoype

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_pheno_23_cm.csv')

image_ids = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt.csv')['image_id'].tolist()

# filter individuals
hip_pheno = hip_pheno[hip_pheno['image_id'].isin(image_ids)]

# new phenotypes
hip_pheno['leg_length'] = (hip_pheno['leg_123_left'] + hip_pheno['leg_123_right']) / 2
add_pheno = hip_pheno[['eid', 'leg_length']]

# process male

# merge previous phenotypes
hip_pheno_gwas_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_231025.txt', sep = " ")
hip_pheno_gwas_male = hip_pheno_gwas_male.merge(add_pheno, left_on='FID', right_on='eid', how='inner')
hip_pheno_gwas_male.drop(columns=['eid'], inplace=True)

hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_240104.txt', index=False, sep = " ")
hip_pheno_gwas_male.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_no_header_240104.txt', index=False, header=False, sep = " ")

# process female

# merge previous phenotypes
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_231025.txt', sep = " ")
hip_pheno_gwas_female = hip_pheno_gwas_female.merge(add_pheno, left_on='FID', right_on='eid', how='inner')
hip_pheno_gwas_female.drop(columns=['eid'], inplace=True)

hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_240104.txt', index=False, sep = " ")
hip_pheno_gwas_female.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_no_header_240104.txt', index=False, header=False, sep = " ")

In [ ]:
hip_pheno_gwas_male.shape

In [ ]:
hip_pheno_gwas_female.shape

#### gwas for female other skeletal traits

In [ ]:
female_skeletal = pd.read_csv("key_results/hip_select_pheno_cm_eid_flt_z_flt_female.csv"); female_skeletal

In [ ]:
female_skeletal.columns

In [ ]:
pheno = ['eid', 'standing_height_imaging_visit', 'head_diameter', 'trochanter_distance', 'shoulder_width',
         'torso_length', 'humerus', 'femur', 'forearm', 'tibia']

# male dataframe for gwas preparation
female_skeletal = female_skeletal[pheno]

female_skeletal_gwas = female_skeletal.iloc[:, 1:]

ids = female_skeletal['eid'].tolist()

female_skeletal_gwas['FID'] = ids
female_skeletal_gwas['IID'] = ids

# put FID and IID to the first two columns
cols = female_skeletal_gwas.columns
cols = cols[-2:].tolist() + cols[:-2].tolist()
female_skeletal_gwas = female_skeletal_gwas[cols]

In [ ]:
female_skeletal_gwas

In [ ]:
female_skeletal_gwas[['FID', 'IID', 'standing_height_imaging_visit']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/female_height_gwas_cm_240310.txt', index=False, sep = " ")
female_skeletal_gwas.drop(columns = ['standing_height_imaging_visit']).to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/female_skeletal_gwas_cm_240310.txt', index=False, sep = " ")

#### gwas for male other skeletal traits

In [ ]:
male_skeletal = pd.read_csv("key_results/hip_select_pheno_cm_eid_flt_z_flt_male.csv"); male_skeletal

In [ ]:
pheno = ['eid', 'standing_height_imaging_visit', 'head_diameter', 'trochanter_distance', 'shoulder_width',
         'torso_length', 'humerus', 'femur', 'forearm', 'tibia']

# male dataframe for gwas preparation
male_skeletal = male_skeletal[pheno]

male_skeletal_gwas = male_skeletal.iloc[:, 1:]

ids = male_skeletal['eid'].tolist()

male_skeletal_gwas['FID'] = ids
male_skeletal_gwas['IID'] = ids

# put FID and IID to the first two columns
cols = male_skeletal_gwas.columns
cols = cols[-2:].tolist() + cols[:-2].tolist()
male_skeletal_gwas = male_skeletal_gwas[cols]

male_skeletal_gwas

In [ ]:
male_skeletal_gwas[['FID', 'IID', 'standing_height_imaging_visit']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/male_height_gwas_cm_240318.txt', index=False, sep = " ")
male_skeletal_gwas.drop(columns = ['standing_height_imaging_visit']).to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/male_skeletal_gwas_cm_240318.txt', index=False, sep = " ")

#### get male and female no imaging eids in 400k population

In [ ]:
all_eids = pd.read_csv("UKB_Imaging_Genetics/GWAS_INPUT_DATA/geno_qc_eids_2022-07-28.txt", header=None, sep = ' ')[0].tolist()

len(all_eids)

In [ ]:
fid_info = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info.csv")[['eid', 'sex']]; fid_info

In [ ]:
fid_info = fid_info[fid_info['eid'].isin(all_eids)]

In [ ]:
fid_info.dropna(inplace=True); fid_info

In [ ]:
fid_male = fid_info[fid_info['sex'] == 1]
fid_female = fid_info[fid_info['sex'] == 0]

male_imaging_eids = hip_pheno_male['eid'].tolist()
female_imaging_eids = hip_pheno_female['eid'].tolist()

fid_male_no_img = fid_male[~fid_male['eid'].isin(male_imaging_eids)]
fid_female_no_img = fid_female[~fid_female['eid'].isin(female_imaging_eids)]

In [ ]:
fid_male_no_img[['eid', 'eid']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/eids_male_400k_no_img.txt', index=False, header=False, sep=' ')
fid_female_no_img[['eid', 'eid']].to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/eids_female_400k_no_img.txt', index=False, header=False, sep=' ')

### divide to train and test dataset

In [ ]:
hip_pheno_gwas_male = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_cm_230713.txt', sep = " ")
hip_pheno_gwas_female = pd.read_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_cm_230713.txt', sep = " ")

In [ ]:
hip_pheno_gwas_male

In [ ]:
# male

random.seed(42)

male_eids = hip_pheno_gwas_male['IID'].tolist()
random.shuffle(male_eids)

# Split the list into two parts
male_eids_train = male_eids[:10000]  # First 12000 elements
male_eids_test = male_eids[10000:]  # Remaining elements

hip_pheno_gwas_male_train = hip_pheno_gwas_male[hip_pheno_gwas_male['FID'].isin(male_eids_train)]
hip_pheno_gwas_male_test = hip_pheno_gwas_male[hip_pheno_gwas_male['FID'].isin(male_eids_test)]

hip_pheno_gwas_male_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_train_cm_230801.txt', index=False, sep = " ")
hip_pheno_gwas_male_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_train_cm_no_header_230801.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_male_test.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_test_cm_230801.txt', index=False, sep = " ")
hip_pheno_gwas_male_test.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_male_test_cm_no_header_230801.txt', index=False, header=False, sep = " ")

pd.DataFrame({"FID": male_eids_train, "IID": male_eids_train}).to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_male_train.txt', index=False, header=False, sep=' ')
pd.DataFrame({"FID": male_eids_test, "IID": male_eids_test}).to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_male_test.txt', index=False, header=False, sep=' ')

print(f"Male train dataframe shape: {hip_pheno_gwas_male_train.shape}")
print(f"Male test dataframe shape: {hip_pheno_gwas_male_test.shape}")


# female
female_eids = hip_pheno_gwas_female['IID'].tolist()
random.shuffle(female_eids)

# Split the list into two parts
female_eids_train = female_eids[:10000]  # First 12000 elements
female_eids_test = female_eids[10000:]  # Remaining elements

hip_pheno_gwas_female_train = hip_pheno_gwas_female[hip_pheno_gwas_female['FID'].isin(female_eids_train)]
hip_pheno_gwas_female_test = hip_pheno_gwas_female[hip_pheno_gwas_female['FID'].isin(female_eids_test)]

hip_pheno_gwas_female_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_train_cm_230801.txt', index=False, sep = " ")
hip_pheno_gwas_female_train.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_train_cm_no_header_230801.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_female_test.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_test_cm_230801.txt', index=False, sep = " ")
hip_pheno_gwas_female_test.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_gwas_female_test_cm_no_header_230801.txt', index=False, header=False, sep = " ")

pd.DataFrame({"FID": female_eids_train, "IID": female_eids_train}).to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_female_train.txt', index=False, header=False, sep=' ')
pd.DataFrame({"FID": female_eids_test, "IID": female_eids_test}).to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_select_pheno_eids_female_test.txt', index=False, header=False, sep=' ')

print(f"Female train dataframe shape: {hip_pheno_gwas_female_train.shape}")
print(f"Female test dataframe shape: {hip_pheno_gwas_female_test.shape}")

#### Generate categorical covar for male and female

In [ ]:
covar = pd.read_csv("UKB_Imaging_Genetics/GCTA_INPUT_DATA/cat_covar_all_eid_2021-10-25_no_header.txt", sep = " ", header = None)

In [ ]:
covar[[0, 1, 3]].to_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/cat_covar_all_eid_2021-10-25_no_header_for_single_sex.txt', index=False, header=False, sep = " ")

In [ ]:
covar[[0, 1, 3]]

#### Divide into 2 parts of tables

In [ ]:
cols = hip_pheno_gwas.columns.tolist()[2:]

cols_part1 = cols[:len(cols)//2]
cols_part2 = cols[len(cols)//2:]

cols_part1 = ['FID', 'IID'] + cols_part1
cols_part2 = ['FID', 'IID'] + cols_part2

hip_pheno_gwas_part1 = hip_pheno_gwas[cols_part1]
hip_pheno_gwas_part2 = hip_pheno_gwas[cols_part2]

hip_pheno_gwas_part1.to_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/hip_pheno_gwas_part1.txt', index=False, sep = " ")
hip_pheno_gwas_part1.to_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/hip_pheno_gwas_part1_no_header.txt', index=False, header=False, sep = " ")

hip_pheno_gwas_part2.to_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/hip_pheno_gwas_part2.txt', index=False, sep = " ")
hip_pheno_gwas_part2.to_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/hip_pheno_gwas_part2_no_header.txt', index=False, header=False, sep = " ")

In [ ]:
hip_pheno_gwas_part1

In [ ]:
hip_pheno_gwas_part2

### Double check with Eucharist result

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm_eid_flt_z_flt.csv'); hip_pheno

In [ ]:
# use ratio
hip_pheno['shoulder_width_ratio'] = hip_pheno['shoulder_width'] / hip_pheno['standing_height']
hip_pheno['trochanter_left2trochanter_right_ratio'] = hip_pheno['trochanter_left2trochanter_right'] / hip_pheno['standing_height']
hip_pheno = hip_pheno[['eid', 'shoulder_width_ratio', 'trochanter_left2trochanter_right_ratio']]

ids = hip_pheno['eid'].tolist()

hip_pheno['FID'] = ids
hip_pheno['IID'] = ids

hip_pheno = hip_pheno[['FID', "IID", "shoulder_width_ratio", "trochanter_left2trochanter_right_ratio"]]

In [ ]:
hip_pheno

In [ ]:
hip_pheno.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_pheno_ratio_check.txt', index=False, sep = " ")
hip_pheno.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_pheno_ratio_check_no_header.txt', index=False, header=False, sep = " ")

## Get h2g

### Get h2g from GCTA

In [ ]:
# both
vg2vp = pd.DataFrame(columns=['pheno', 'var', 'se'])
path = "UKB_Imaging_Genetics/GCTA_GRM_OUTPUT/nonimputed_hip_both_hapmap3_20230918"

for file in os.listdir(path):
    if file.endswith(".hsq"):
        # get phenotype name
        pheno = file.split('.')[0]
        if pheno == '':
            continue
        with open(os.path.join(path, file), "r") as f:
            for line in f:
                # get V(G)/Vp
                if line.startswith("V(G)/Vp"):
                    var = line.split()[1]
                    se = line.split()[2]
                    vg2vp = vg2vp.append({'pheno': pheno, 'var': var, 'se': se}, ignore_index=True)

vg2vp = vg2vp.sort_values(by=['var'], ascending=False)
vg2vp.reset_index(drop=True, inplace=True)
vg2vp[['var', 'se']] = vg2vp[['var', 'se']].apply(pd.to_numeric)
vg2vp.to_csv("key_results/h2g/h2g_both_20230918.csv", index=False)

# male
vg2vp_male = pd.DataFrame(columns=['pheno', 'var', 'se'])
path = "UKB_Imaging_Genetics/GCTA_GRM_OUTPUT/nonimputed_hip_male_hapmap3_20230918"

for file in os.listdir(path):
    if file.endswith(".hsq"):
        # get phenotype name
        pheno = file.split('.')[0]
        if pheno == '':
            continue
        with open(os.path.join(path, file), "r") as f:
            for line in f:
                # get V(G)/Vp
                if line.startswith("V(G)/Vp"):
                    var = line.split()[1]
                    se = line.split()[2]
                    vg2vp_male = vg2vp_male.append({'pheno': pheno, 'var': var, 'se': se}, ignore_index=True)

vg2vp_male = vg2vp_male.sort_values(by=['var'], ascending=False)
vg2vp_male.reset_index(drop=True, inplace=True)
vg2vp_male[['var', 'se']] = vg2vp_male[['var', 'se']].apply(pd.to_numeric)
vg2vp_male.to_csv("key_results/h2g/h2g_male_20230918.csv", index=False)

# female
vg2vp_female = pd.DataFrame(columns=['pheno', 'var', 'se'])
path = "UKB_Imaging_Genetics/GCTA_GRM_OUTPUT/nonimputed_hip_female_hapmap3_20230918"

for file in os.listdir(path):
    if file.endswith(".hsq"):
        # get phenotype name
        pheno = file.split('.')[0]
        if pheno == '':
            continue
        with open(os.path.join(path, file), "r") as f:
            for line in f:
                # get V(G)/Vp
                if line.startswith("V(G)/Vp"):
                    var = line.split()[1]
                    se = line.split()[2]
                    vg2vp_female = vg2vp_female.append({'pheno': pheno, 'var': var, 'se': se}, ignore_index=True)

vg2vp_female = vg2vp_female.sort_values(by=['var'], ascending=False)
vg2vp_female.reset_index(drop=True, inplace=True)
vg2vp_female[['var', 'se']] = vg2vp_female[['var', 'se']].apply(pd.to_numeric)
vg2vp_female.to_csv("key_results/h2g/h2g_female_20230918.csv", index=False)

In [ ]:
# concatenate
vg2vp['sex'] = "both"
vg2vp_male['sex'] = "male"
vg2vp_female['sex'] = "female"

df = pd.concat([vg2vp, vg2vp_male, vg2vp_female])
df.to_csv("key_results/h2g/h2g_all_20230918.csv", index=False)

In [ ]:
df

### Get h2g from LDSC

In [ ]:
path = "/Users/alexxu/Library/CloudStorage/Box-Box/Narasimhan_lab/hip_shape/UKB_Imaging_Genetics/LDSC_OUTPUT"

In [ ]:
group = ['both', 'female', 'male']

h2g_df = pd.DataFrame()

for sex in group:
    folder_name = f"h2g_hip_{group}_20230909"
    for file_name in os.listdir(os.path.join(path, folder_name)):
        if file_name.endswith(".log"):
            pheno = file_name.split('.')[0]
            with open(os.path.join(path, folder_name, file_name), "r") as f:
                for line in f:
                    if line.startswith("Total Observed scale h2"):
                        h2g = float(line.split(":")[-1].strip().split()[0])
                        se = float(line.split(":")[-1].strip().split()[1].replace("(" , "").replace(")", ""))
                        h2g_df = h2g_df.append({'pheno': pheno, 'h2g': h2g, 'se': se, 'sex': sex}, ignore_index=True)
                        break
h2g_df.to_csv("key_results/h2g/h2g_ldsc_all_20230909.csv", index=False)

In [ ]:
h2g_df

## GWAS

#### Try with PLINK

In [ ]:
hip_pheno_gwas = pd.read_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/hip_pheno_gwas.txt', sep=" "); hip_pheno_gwas

In [ ]:
hip_height_gwas = hip_pheno_gwas[['FID', 'IID', 'hip_height']]; hip_height_gwas

In [ ]:
hip_height_gwas.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_height_gwas.txt', index=False, sep = " ")

### Using BOLT-LMM ~2:30min per trait

In [ ]:
hip_pheno_gwas = pd.read_csv('UKB_Imaging_Genetics/GCTA_INPUT_DATA/hip_pheno_gwas.txt', sep=" "); hip_pheno_gwas

In [ ]:
# devide phenotypes into 4 parts each part has 70 phenotypes, all parts should contain first 2 columns (FID, IID)

id_col = hip_pheno_gwas.columns[:2].tolist()
col_part1 = hip_pheno_gwas.columns[2:72].tolist()
col_part2 = hip_pheno_gwas.columns[72:142].tolist()
col_part3 = hip_pheno_gwas.columns[142:212].tolist()
col_part4 = hip_pheno_gwas.columns[212:].tolist()

hip_pheno_gwas_part1 = pd.concat([hip_pheno_gwas[id_col], hip_pheno_gwas[col_part1]], axis=1)
hip_pheno_gwas_part2 = pd.concat([hip_pheno_gwas[id_col], hip_pheno_gwas[col_part2]], axis=1)
hip_pheno_gwas_part3 = pd.concat([hip_pheno_gwas[id_col], hip_pheno_gwas[col_part3]], axis=1)
hip_pheno_gwas_part4 = pd.concat([hip_pheno_gwas[id_col], hip_pheno_gwas[col_part4]], axis=1)

hip_pheno_gwas_part1.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_pheno_gwas_part1.txt', index=False, sep = " ")
hip_pheno_gwas_part2.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_pheno_gwas_part2.txt', index=False, sep = " ")
hip_pheno_gwas_part3.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_pheno_gwas_part3.txt', index=False, sep = " ")
hip_pheno_gwas_part4.to_csv('UKB_Imaging_Genetics/GWAS_INPUT_DATA/hip_pheno_gwas_part4.txt', index=False, sep = " ")

### Use drinking as an nagative control

In [ ]:
hip_pheno_gwas = pd.read_csv("key_results/hip_pheno_23_eid_flt.csv"); hip_pheno_gwas

In [ ]:
# fid for Average weekly red wine intake
drinking_fid = pd.read_csv("gwas/fid1568.csv")[['eid', '1568-0.0']]; drinking_fid

In [ ]:
hip_pheno_gwas

In [ ]:
hip_pheno_gwas_eid =  hip_pheno_gwas[['Patient EID']]
neg_pheno_gwas = hip_pheno_gwas_eid.merge(drinking_fid, left_on='Patient EID', right_on='eid', how='inner').rename(columns={'Patient EID': "FID",
                                                                                                                            'eid': "IID", 
                                                                                                                            '1568-0.0': 'drinking'}); neg_pheno_gwas

In [ ]:
neg_pheno_gwas.dropna(inplace=True)

In [ ]:
neg_pheno_gwas.to_csv("key_results/neg_pheno_gwas.txt", index=False, sep=" ")

### Clumped SNPs

#### for both sexes, male, and female separately

In [ ]:
path = 'UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/'

clump_both = pd.DataFrame()

for file in os.listdir(os.path.join(path, 'hip_both_1000_noverbose')):
    if file.endswith(".ranges"):
        tmp_df = pd.read_csv(os.path.join(path, 'hip_both_1000_noverbose', file), sep="\s+")
        pheno_name = file.split(".")[1]
        tmp_df['pheno'] = pheno_name
        if clump_both.empty:
            clump_both = tmp_df
        else:
            clump_both = pd.concat([clump_both, tmp_df], axis=0)

clump_male = pd.DataFrame()

for file in os.listdir(os.path.join(path, 'hip_male_1000_noverbose')):
    if file.endswith(".ranges"):
        tmp_df = pd.read_csv(os.path.join(path, 'hip_male_1000_noverbose', file), sep="\s+")
        pheno_name = file.split(".")[1]
        tmp_df['pheno'] = pheno_name
        if clump_male.empty:
            clump_male = tmp_df
        else:
            clump_male = pd.concat([clump_male, tmp_df], axis=0)

clump_female = pd.DataFrame()

for file in os.listdir(os.path.join(path, 'hip_female_1000_noverbose')):
    if file.endswith(".ranges"):
        tmp_df = pd.read_csv(os.path.join(path, 'hip_female_1000_noverbose', file), sep="\s+")
        pheno_name = file.split(".")[1]
        tmp_df['pheno'] = pheno_name
        if clump_female.empty:
            clump_female = tmp_df
        else:
            clump_female = pd.concat([clump_female, tmp_df], axis=0)

In [ ]:
clump_both.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both.txt", index=False, sep=" ")
clump_female.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_female.txt", index=False, sep=" ")
clump_male.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_male.txt", index=False, sep=" ")

In [ ]:
clump_both[['SNP']].to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_snp.txt", index=False, sep=" ", header=False)
clump_female[['SNP']].to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_female_snp.txt", index=False, sep=" ", header=False)
clump_male[['SNP']].to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_male_snp.txt", index=False, sep=" ", header=False)

#### after doing pruning, merge the pruned snps with chr position

In [ ]:
# get pruned snps

pruned_both = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_snp.prune.in", sep="\s+", header=None)
pruned_female = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_female_snp.prune.in", sep="\s+", header=None)
pruned_male = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_male_snp.prune.in", sep="\s+", header=None)

pruned_both_snps = pruned_both[0].tolist()
pruned_female_snps = pruned_female[0].tolist()
pruned_male_snps = pruned_male[0].tolist()

clump_both_pruned = clump_both[clump_both['SNP'].isin(pruned_both_snps)]
clump_female_pruned = clump_female[clump_female['SNP'].isin(pruned_female_snps)]
clump_male_pruned = clump_male[clump_male['SNP'].isin(pruned_male_snps)]

print(f"clump_both: {clump_both['SNP'].unique().shape[0]}, clump_both_pruned: {clump_both_pruned['SNP'].unique().shape[0]}")
print(f"clump_female: {clump_female['SNP'].unique().shape[0]}, clump_female_pruned: {clump_female_pruned['SNP'].unique().shape[0]}")
print(f"clump_male: {clump_male['SNP'].unique().shape[0]}, clump_male_pruned: {clump_male_pruned['SNP'].unique().shape[0]}")

In [ ]:
clump_both_pruned.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_pruned.csv", index=False,)
clump_female_pruned.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_female_pruned.csv", index=False)
clump_male_pruned.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_male_pruned.csv", index=False)

In [ ]:
# get pruned snps

pruned_both = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_snp_vif1.1.prune.in", sep="\s+", header=None)
pruned_female = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_female_snp_vif1.1.prune.in", sep="\s+", header=None)
pruned_male = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_male_snp_vif1.1.prune.in", sep="\s+", header=None)

pruned_both_snps = pruned_both[0].tolist()
pruned_female_snps = pruned_female[0].tolist()
pruned_male_snps = pruned_male[0].tolist()

clump_both_pruned = clump_both[clump_both['SNP'].isin(pruned_both_snps)]
clump_female_pruned = clump_female[clump_female['SNP'].isin(pruned_female_snps)]
clump_male_pruned = clump_male[clump_male['SNP'].isin(pruned_male_snps)]

print(f"clump_both: {clump_both['SNP'].unique().shape[0]}, clump_both_pruned: {clump_both_pruned['SNP'].unique().shape[0]}")
print(f"clump_female: {clump_female['SNP'].unique().shape[0]}, clump_female_pruned: {clump_female_pruned['SNP'].unique().shape[0]}")
print(f"clump_male: {clump_male['SNP'].unique().shape[0]}, clump_male_pruned: {clump_male_pruned['SNP'].unique().shape[0]}")

In [ ]:
clump_both_pruned

In [ ]:
clump_both_pruned.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_pruned_vif1.1.csv", index=False,)
clump_female_pruned.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_female_pruned_vif1.1.csv", index=False)
clump_male_pruned.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_male_pruned_vif1.1.csv", index=False)

In [ ]:
df = pd.read_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_pruned_vif1.1.csv")

# Sort the dataframe by chromosome and start position
df['POS'] = df['POS'].apply(lambda x: x.split(':')[1])
df['start'], df['end'] = df['POS'].apply(lambda x: x.split('..')[0]), df['POS'].apply(lambda x: x.split('..')[1])
df[['start', 'end']] = df[['start', 'end']].apply(pd.to_numeric)
df.sort_values(['CHR', 'start'], inplace=True)

# Calculate overlaps
overlaps = []
for chr_id in df['CHR'].unique():
    chr_data = df[df['CHR'] == chr_id]
    for i in range(len(chr_data) - 1):
        for j in range(i + 1, len(chr_data)):
            if chr_data.iloc[i]['SNP'] != chr_data.iloc[j]['SNP']:
                start1, end1 = chr_data.iloc[i][['start', 'end']]
                start2, end2 = chr_data.iloc[j][['start', 'end']]
                overlap = max(0, min(end1, end2) - max(start1, start2))
                overlaps.append({'CHR': chr_id, 'SNP1': chr_data.iloc[i]['SNP'], 'SNP2': chr_data.iloc[j]['SNP'], 'Overlap': overlap})

# Convert to a dataframe
overlaps_df = pd.DataFrame(overlaps)
overlaps_df.sort_values('Overlap', ascending=False, inplace=True)
overlaps_df.drop_duplicates(subset=['SNP1', 'SNP2'], inplace=True)
overlaps_df.to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_pruned_vif1.1_overlaps.csv", index=False)

In [ ]:
overlaps_df

In [ ]:
overlaps_df[['SNP1', 'SNP2']].to_csv("UKB_Imaging_Genetics/PLINK_CLUMP_OUTPUT/hip_1000_noverbose_summary/clump_both_pruned_vif1.1_overlaps_snps.txt", index=False, sep = ' ', header=False)

In [ ]:
unique_snp_male = clump_male['SNP'].unique()
unique_snp_female = clump_female['SNP'].unique()
unique_snp_both = clump_both['SNP'].unique()

fm_overlap = np.intersect1d(unique_snp_male, unique_snp_female)
both_fm_overlap = [i for i in unique_snp_both if i in unique_snp_male or i in unique_snp_female]

In [ ]:
len(unique_snp_both)

In [ ]:
len(unique_snp_male)

In [ ]:
len(unique_snp_female)

In [ ]:
def get_one_gene_snp(range):
    list_element = range.replace('[', '').replace(']', '').split(',')
    if len(list_element) == 1 and list_element[0] != '':
        return True
    return False

In [ ]:
clump_male_one_gene = clump_male[clump_male['RANGES'].apply(lambda x: get_one_gene_snp(x))]
clump_female_one_gene = clump_female[clump_female['RANGES'].apply(lambda x: get_one_gene_snp(x))]

In [ ]:
clump_male_one_gene

In [ ]:
{
    "CASC20": "heterotopic ossification",
    "FBN1": "Marfan syndrome",
    "SRBD1": "Glaucoma",
    "BNC2": "adolescent idiopathic scoliosis",
    "CCDC91": "ossification of the posterior longitudinal ligament of the spine",
    "EYA2": "hypaxial somitic myogenesis in the mouse embryo",
    "DIS3L2": "perlman syndrome",
    "ARHGAP22": "type 2 diabetic retinopathy",
}

In [ ]:
clump_female_one_gene

# Genetic correlation

## Cross phenotypes rg

In [ ]:
!mkdir -p key_results/genetic_correlation/rg_20230909

In [ ]:
# both rg
rg_df_both = pd.DataFrame(columns=['pheno1', 'pheno1_h2g', 'pheno1_h2g_se', 'pheno1_lambda',
                                   'pheno2', 'pheno2_h2g', 'pheno2_h2g_se', 'pheno2_lambda',
                                   'cor', 'se', 'z', 'p_value'])
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_hip_both_20230909"
for file in os.listdir(path):
    if file.endswith(".log"):
        # get phenotype name
        pheno1, pheno2 = file.split('-')
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('.')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)

        rg_df_both = rg_df_both.append({'pheno1': pheno1, 'pheno1_h2g': h2g_1, 'pheno1_h2g_se': h2g_1_se, 'pheno1_lambda': pheno1_lambda,
                                        'pheno2': pheno2, 'pheno2_h2g': h2g_2, 'pheno2_h2g_se': h2g_2_se, 'pheno2_lambda': pheno2_lambda,
                                        'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                                        ignore_index=True)

rg_df_both.sort_values(by = ['pheno1', 'pheno2'], inplace=True)
rg_df_both['gender'] = 'both'
rg_df_both.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_both.csv", index=False)

# female rg
rg_df_female = pd.DataFrame(columns=['pheno1', 'pheno1_h2g', 'pheno1_h2g_se', 'pheno1_lambda',
                                   'pheno2', 'pheno2_h2g', 'pheno2_h2g_se', 'pheno2_lambda',
                                   'cor', 'se', 'z', 'p_value'])
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_hip_female_20230909"
for file in os.listdir(path):
    if file.endswith(".log"):
        # get phenotype name
        pheno1, pheno2 = file.split('-')
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('.')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)

        rg_df_female = rg_df_female.append({'pheno1': pheno1, 'pheno1_h2g': h2g_1, 'pheno1_h2g_se': h2g_1_se, 'pheno1_lambda': pheno1_lambda,
                                          'pheno2': pheno2, 'pheno2_h2g': h2g_2, 'pheno2_h2g_se': h2g_2_se, 'pheno2_lambda': pheno2_lambda,
                                          'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                                          ignore_index=True)

rg_df_female.sort_values(by = ['pheno1', 'pheno2'], inplace=True)
rg_df_female['gender'] = 'female'
rg_df_female.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_female.csv", index=False)

# male rg
rg_df_male = pd.DataFrame(columns=['pheno1', 'pheno1_h2g', 'pheno1_h2g_se', 'pheno1_lambda',
                                   'pheno2', 'pheno2_h2g', 'pheno2_h2g_se', 'pheno2_lambda',
                                   'cor', 'se', 'z', 'p_value'])
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_hip_male_20230909"
for file in os.listdir(path):
    if file.endswith(".log"):
        # get phenotype name
        pheno1, pheno2 = file.split('-')
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('.')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)

        rg_df_male = rg_df_male.append({'pheno1': pheno1, 'pheno1_h2g': h2g_1, 'pheno1_h2g_se': h2g_1_se, 'pheno1_lambda': pheno1_lambda,
                                          'pheno2': pheno2, 'pheno2_h2g': h2g_2, 'pheno2_h2g_se': h2g_2_se, 'pheno2_lambda': pheno2_lambda,
                                          'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                                          ignore_index=True)

rg_df_male.sort_values(by = ['pheno1', 'pheno2'], inplace=True)
rg_df_male['gender'] = 'male'
rg_df_male.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_male.csv", index=False)

In [ ]:
rg_df_male

### Prepare for both rg plot

In [ ]:
rg_df_both['pheno1'].value_counts()

In [ ]:
rg_df_both['pheno2'].value_counts()

## Male vs Female rg

In [ ]:
# male vs female rg
rg_df_fvsm = pd.DataFrame(columns=['pheno_f', 'pheno_f_h2g', 'pheno_f_h2g_se', 'pheno_f_lambda',
                                   'pheno_m', 'pheno_m_h2g', 'pheno_m_h2g_se', 'pheno_m_lambda',
                                   'cor', 'se', 'z', 'p_value'])
# path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_hip_female_vs_male_20230909"
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_hip_female_vs_male_20240111"
for file in os.listdir(path):
    if file.endswith(".log"):
        # get phenotype name
        pheno1, pheno2 = file.split('-')
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('.')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)

        rg_df_fvsm = rg_df_fvsm.append({'pheno_f': pheno1, 'pheno_f_h2g': h2g_1, 'pheno_f_h2g_se': h2g_1_se, 'pheno_f_lambda': pheno1_lambda,
                                        'pheno_m': pheno2, 'pheno_m_h2g': h2g_2, 'pheno_m_h2g_se': h2g_2_se, 'pheno_m_lambda': pheno2_lambda,
                                        'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                                        ignore_index=True)

rg_df_fvsm.sort_values(by = ['pheno_f', 'pheno_m'], inplace=True)
rg_df_fvsm = rg_df_fvsm[rg_df_fvsm['pheno_f'] == rg_df_fvsm['pheno_m']]

# add long bone male vs female correlation
rg_df_fvsm['pheno'] = rg_df_fvsm['pheno_f']
rg_df_fvsm = rg_df_fvsm[['pheno', 'cor', 'se']]

long_bone_cor = pd.DataFrame({
    'pheno': ['Humerus', 'Forearm', 'Torso length', 'Femur', 'Shoulder width', 'Tibia'],
    'cor': [0.99, 0.97, 0.96, 0.95, 0.94, 0.92],
    'se': [0.04, 0.04, 0.05, 0.04, 0.05, 0.04]
})

rg_df_fvsm = pd.concat([rg_df_fvsm, long_bone_cor]).reset_index(drop=True)
rg_df_fvsm.to_csv("key_results/genetic_correlation/rg_hip_female_vs_male_240112.csv", index=False)

## Combine male and female genetic correlation

In [ ]:
male_rg = pd.read_csv("key_results/genetic_correlation/rg_20230909/rg_hip_male.csv")
female_rg = pd.read_csv("key_results/genetic_correlation/rg_20230909/rg_hip_female.csv")

In [ ]:
male_rg['pheno1'].value_counts()

In [ ]:
male_rg['pheno2'].value_counts()

In [ ]:
male_rg

In [ ]:
female_rg

In [ ]:
female_rg.rename(columns={'pheno1': 'pheno2', 'pheno2': 'pheno1'}, inplace=True)

all_cor = pd.concat([male_rg, 
                    female_rg], axis=0)

In [ ]:
all_cor

In [ ]:
for pheno in all_cor.pheno1.unique():
    cor = {'pheno1': pheno, 'pheno2': pheno, 'cor': 1, 'p_value': 0, 'type': np.nan}
    all_cor = all_cor.append(cor, ignore_index=True)

all_cor

In [ ]:
all_cor.to_csv("key_results/genetic_correlation/male_vs_female_genetic_cor.csv", index=False)

## Combine genetic correlation and phenotypic correlation

In [ ]:
# get genetic correlation
rg_df_both = pd.read_csv("key_results/genetic_correlation/rg_20230909/rg_hip_both.csv")

In [ ]:
# phenotype correlation
hip_pheno_flt = pd.read_csv('key_results/hip_select_pheno_cm_residual.csv').drop(columns=['standing_height'])
hip_pheno_flt = hip_pheno_flt.iloc[:, 1:]

pheno_cor = pd.DataFrame(columns=['pheno1', 'pheno2', 'cor', 'p_value', 'neglog10p'])

for col1 in hip_pheno_flt.columns:
    for col2 in hip_pheno_flt.columns:
        corr, p_value = pearsonr(hip_pheno_flt[col1], hip_pheno_flt[col2])
        pheno_cor = pheno_cor.append({'pheno1': col1, 
                                      'pheno2': col2, 
                                      'cor': corr, 
                                      'p_value': p_value,
                                      'neglog10p': -np.log10(p_value)},  ignore_index=True)
pheno_cor['type'] = 'phenotype_cor'
pheno_cor = pheno_cor.merge(rg_df_both[['pheno1', 'pheno2']], on=['pheno1', 'pheno2'], how='inner')

pheno_cor

In [ ]:
# merge genetic correlation and phenotype correlation
rg_df_both.rename(columns={'pheno1': 'pheno2', 'pheno2': 'pheno1'}, inplace=True)
rg_df_both['type'] = 'genetic_cor'
all_cor = pd.concat([rg_df_both[['pheno1', 'pheno2', 'cor', 'p_value', 'neglog10p', 'type']], pheno_cor], axis=0)

In [ ]:
all_cor

In [ ]:
for pheno in all_cor.pheno1.unique():
    cor = {'pheno1': pheno, 'pheno2': pheno, 'cor': 1, 'p_value': 0, 'type': np.nan}
    all_cor = all_cor.append(cor, ignore_index=True)

all_cor

In [ ]:
all_cor.to_csv("key_results/genetic_correlation/genetic_and_phenotype_cor.csv", index=False)

In [ ]:
all_cor['pheno2'].value_counts()

### get rg for regression comparation

In [ ]:
# different kinds of regression rg
rg_df = pd.DataFrame(columns=['pheno1', 'pheno2', 'cor', 'se', 'z', 'p_value'])
path = "key_results/genetic_correlation/rg_20230626"
for file in os.listdir(path):
    if file.endswith(".log"):
        # get phenotype name
        pheno1, pheno2 = file.split('-')
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('.')[0]

        with open(os.path.join(path, file), "r") as f:
            for line in f:
                # get rg info
                if line.startswith("Genetic Correlation:"):
                    # print(line)
                    rg = line.split(" ")[2].strip()
                    rg = float(rg)
                    # print(rg)
                    se = line.split(" ")[3].strip()
                    se = se[1:-1]
                    se = float(se)
                if line.startswith("Z-score:"):
                    z = line.split(" ")[1].strip()
                    z = float(z)
                if line.startswith("P:"):
                    p_value = line.split(" ")[1].strip()
                    p_value = float(p_value)
                    neglog10p = -np.log10(p_value)

            rg_df = rg_df.append({'pheno1': pheno1, 'pheno2': pheno2, 'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, ignore_index=True)

rg_df.sort_values(by = ['pheno1', 'pheno2'], inplace=True)
rg_df.to_csv("key_results/genetic_correlation/rg_20230626/rg_reg_compare.csv", index=False)


In [ ]:
rg_df

In [ ]:
# Create a pivot table for heatmap
correlation_table = rg_df.pivot('pheno1', 'pheno2', 'cor')

# Create a pivot table for p-values
p_value_table = rg_df.pivot('pheno1', 'pheno2', 'p_value')

# Create a new DataFrame to hold the display values
display_table = correlation_table.astype(str) + "\n(p=" + p_value_table.round(3).astype(str) + ")"

# Plotting
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_table, annot=display_table, fmt='', cmap="RdBu_r", center=0)
plt.title("Genetic Correlation Heatmap")
plt.xlabel("")
plt.ylabel("")
plt.show()

## Genetic correlation for obstetrical phenotypes and pelvic shape

#### Female

In [ ]:
!mkdir -p key_results/genetic_correlation/rg_20230909

In [ ]:
df = pd.DataFrame(columns=['pheno_pelvic', 'pheno_pelvic_h2g', 'pheno_pelvic_h2g_se', 'pheno_pelvic_lambda',
                           'pheno_outcome', 'pheno_outcome_h2g', 'pheno_outcome_h2g_se', 'pheno_outcome_lambda',
                           'cor', 'se', 'z', 'p_value'])

path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_pheno_outcome_20230909/female/"

for file in os.listdir(path):
    if file.endswith(".log"):
        file_modified = file.replace("_-_", "_")
        # get phenotype name
        pheno1 = file_modified.split('-')[0]
        pheno2 = file_modified.split('-')[1]
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('_rg.log')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)
            if line.startswith("ERROR"):
                rg = "NA"
                se = "NA"
                z = "NA"
                p_value = "NA"
                neglog10p = "NA"
                break

        df = df.append({'pheno_pelvic': pheno1, 'pheno_pelvic_h2g': h2g_1, 'pheno_pelvic_h2g_se': h2g_1_se, 'pheno_pelvic_lambda': pheno1_lambda,
                        'pheno_outcome': pheno2, 'pheno_outcome_h2g': h2g_2, 'pheno_outcome_h2g_se': h2g_2_se, 'pheno_outcome_lambda': pheno2_lambda,
                        'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                        ignore_index=True)

df.sort_values(by = ['pheno_pelvic', 'pheno_outcome'], inplace=True)
df.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_pelvic_vs_outcome_female.csv", index=False)

In [ ]:
df[df['pheno_outcome'] == "finngen_R9_O15_LABOUR_PELVIC_ABNORM"]

#### Male

In [ ]:
df = pd.DataFrame(columns=['pheno_pelvic', 'pheno_pelvic_h2g', 'pheno_pelvic_h2g_se', 'pheno_pelvic_lambda',
                           'pheno_outcome', 'pheno_outcome_h2g', 'pheno_outcome_h2g_se', 'pheno_outcome_lambda',
                           'cor', 'se', 'z', 'p_value'])

path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_pheno_outcome_20230909/male/"

for file in os.listdir(path):
    if file.endswith(".log"):
        file_modified = file.replace("_-_", "_")
        # get phenotype name
        pheno1 = file_modified.split('-')[0]
        pheno2 = file_modified.split('-')[1]
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('_rg.log')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)
            if line.startswith("ERROR"):
                rg = "NA"
                se = "NA"
                z = "NA"
                p_value = "NA"
                neglog10p = "NA"
                break

        df = df.append({'pheno_pelvic': pheno1, 'pheno_pelvic_h2g': h2g_1, 'pheno_pelvic_h2g_se': h2g_1_se, 'pheno_pelvic_lambda': pheno1_lambda,
                        'pheno_outcome': pheno2, 'pheno_outcome_h2g': h2g_2, 'pheno_outcome_h2g_se': h2g_2_se, 'pheno_outcome_lambda': pheno2_lambda,
                        'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                        ignore_index=True)

df.sort_values(by = ['pheno_pelvic', 'pheno_outcome'], inplace=True)
df.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_pelvic_vs_outcome_male.csv", index=False)

#### Both

In [ ]:
df = pd.DataFrame(columns=['pheno_pelvic', 'pheno_pelvic_h2g', 'pheno_pelvic_h2g_se', 'pheno_pelvic_lambda',
                           'pheno_outcome', 'pheno_outcome_h2g', 'pheno_outcome_h2g_se', 'pheno_outcome_lambda',
                           'cor', 'se', 'z', 'p_value'])

path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_pheno_outcome_20230909/both/"

for file in os.listdir(path):
    if file.endswith(".log"):
        file_modified = file.replace("_-_", "_")
        # get phenotype name
        pheno1 = file_modified.split('-')[0]
        pheno2 = file_modified.split('-')[1]
        pheno1 = pheno1.split('.')[0]
        pheno2 = pheno2.split('_rg.log')[0]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)
            if line.startswith("ERROR"):
                rg = "NA"
                se = "NA"
                z = "NA"
                p_value = "NA"
                neglog10p = "NA"
                break

        df = df.append({'pheno_pelvic': pheno1, 'pheno_pelvic_h2g': h2g_1, 'pheno_pelvic_h2g_se': h2g_1_se, 'pheno_pelvic_lambda': pheno1_lambda,
                        'pheno_outcome': pheno2, 'pheno_outcome_h2g': h2g_2, 'pheno_outcome_h2g_se': h2g_2_se, 'pheno_outcome_lambda': pheno2_lambda,
                        'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                        ignore_index=True)

df.sort_values(by = ['pheno_pelvic', 'pheno_outcome'], inplace=True)
df.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_pelvic_vs_outcome_both.csv", index=False)

In [ ]:
df

### Combine male and female together to see if any difference

In [ ]:
female_df = pd.read_csv("key_results/genetic_correlation/rg_20230909/rg_hip_pelvic_vs_outcome_female.csv")
male_df = pd.read_csv("key_results/genetic_correlation/rg_20230909/rg_hip_pelvic_vs_outcome_male.csv")
female_df = female_df[['pheno_pelvic', 'pheno_outcome', 'cor', 'se', 'z', 'p_value']]
male_df = male_df[['pheno_pelvic', 'pheno_outcome', 'cor', 'se', 'z', 'p_value']]
female_df.rename(columns = {'cor': 'female_cor', 'se': 'female_se', 'z': 'female_z', 'p_value': 'female_p'}, inplace = True)
male_df.rename(columns = {'cor': 'male_cor', 'se': 'male_se', 'z': 'male_z', 'p_value': 'male_p'}, inplace = True)

# Merge
mf_df = female_df.merge(male_df, on = ['pheno_pelvic', 'pheno_outcome'], how = 'inner')

In [ ]:
mf_df

In [ ]:
mf_df.to_csv("key_results/genetic_correlation/rg_20230909/rg_hip_pelvic_vs_outcome_male_and_female.csv", index=False)

## rg for Handedness

In [ ]:
!mkdir -p key_results/genetic_correlation/rg_20230830

In [ ]:
# both rg
rg_df = pd.DataFrame(columns=['pheno1', 'pheno1_h2g', 'pheno1_h2g_se', 'pheno1_lambda',
                                   'pheno2', 'pheno2_h2g', 'pheno2_h2g_se', 'pheno2_lambda',
                                   'cor', 'se', 'z', 'p_value'])
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/rg_hip_handedness_20230830"
for file in os.listdir(path):
    if file.endswith(".log"):
        # get phenotype name
        pheno1, pheno2 = file.split('-')
        pheno1 = pheno1.split('.')[1]
        pheno2 = pheno2.split('.')
        if len(pheno2) == 2:
            pheno2 = pheno2[0]
        else:
            pheno2 = pheno2[1]

        with open(os.path.join(path, file), "r") as f:
            lines = f.readlines()cd

        for i, line in enumerate(lines):
            # get h2g
            if line.startswith("Heritability of phenotype 1") and i + 2 < len(lines):
                h2g_1 = lines[i + 2].split(" ")[4].strip()
                h2g_1 = float(h2g_1)

                h2g_1_se = lines[i + 2].split(" ")[5].strip()
                h2g_1_se = h2g_1_se[1:-1]
                h2g_1_se = float(h2g_1_se)

                pheno1_lambda = lines[i + 3].split(" ")[2].strip()
                pheno1_lambda = float(pheno1_lambda)
            
            if line.startswith("Heritability of phenotype 2/2") and i + 2 < len(lines):
                h2g_2 = lines[i + 2].split(" ")[4].strip()
                h2g_2 = float(h2g_2)
                
                h2g_2_se = lines[i + 2].split(" ")[5].strip()
                h2g_2_se = h2g_2_se[1:-1]
                h2g_2_se = float(h2g_2_se)

                pheno2_lambda = lines[i + 3].split(" ")[2].strip()
                pheno2_lambda = float(pheno1_lambda)

            # get rg
            if line.startswith("Genetic Correlation:"):
                # print(line)
                rg = line.split(" ")[2].strip()
                rg = float(rg)
                # print(rg)
                se = line.split(" ")[3].strip()
                se = se[1:-1]
                se = float(se)
            if line.startswith("Z-score:"):
                z = line.split(" ")[1].strip()
                z = float(z)
            if line.startswith("P:"):
                p_value = line.split(" ")[1].strip()
                p_value = float(p_value)
                neglog10p = -np.log10(p_value)

        rg_df = rg_df.append({'pheno1': pheno1, 'pheno1_h2g': h2g_1, 'pheno1_h2g_se': h2g_1_se, 'pheno1_lambda': pheno1_lambda,
                                        'pheno2': pheno2, 'pheno2_h2g': h2g_2, 'pheno2_h2g_se': h2g_2_se, 'pheno2_lambda': pheno2_lambda,
                                        'cor': rg, 'se': se, 'z': z, 'p_value': p_value, 'neglog10p': neglog10p}, 
                                        ignore_index=True)

rg_df.sort_values(by = ['pheno1', 'pheno2'], inplace=True)
rg_df.to_csv("key_results/genetic_correlation/rg_20230830/rg_hip_handedness.csv", index=False)


In [ ]:
rg_df

# SLDSC

## for ATAC-seq from science advance paper

In [ ]:
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/SLDSC_OUTPUT/part_h2g_20230920_sa_atacseq"

In [ ]:
both_res_files = []
male_res_files = [] 
female_res_files = []

for f in os.listdir(path):
    if f.endswith('.results'):
        if "both" in f:
            both_res_files.append(f)
        elif "female" in f:
            female_res_files.append(f)
        elif "male" in f:
            male_res_files.append(f)

print("Both: ", len(both_res_files))
print("Female: ", len(female_res_files))
print("Male: ", len(male_res_files))

In [ ]:
def assign_group(i):
    if "Human" in i and "Brain-Filtered" in i:
        group = "Human_Brain_Filtered"
    elif "Human" in i and "Gain" in i:
        group = "Human_Gain"
    elif "Human" in i and "Mouse" in i and "Shared" in i:
        group = "Human_Mouse_Shared"
    elif "Human" in i and "Mouse" in i and "Intersect" in i:
        group = "Human_Mouse_Intersect"
    elif "Mouse" in i and "Brain-Filter" in i:
        group = "Mouse_Brain_Filtered"
    elif "Mouse" in i and "Gain" in i:
        group = "Mouse_Gain"
    elif "Mouse" in i:
        group = "Mouse"
    elif "Human" in i:
        group = "Human"
    return group

#### For both male and female

In [ ]:
all_res_df = None

for f in both_res_files:
    anno, pheno = f.split('iPSYCH-PGC_ASD_baseline_')[1].split('_both.results')[0].split('_sorted_control_')
    df = pd.read_csv(os.path.join(path, f), sep='\t')
    df = df[df['Category'] == 'L2_1']
    df['Annotation'] = anno
    df['Phenotype'] = pheno

    if all_res_df is None:
        all_res_df = df
    else:
        all_res_df = pd.concat([all_res_df, df], axis=0)

all_res_df['Group'] = all_res_df['Annotation'].apply(lambda x: assign_group(x))
all_res_df['neglog10p'] = -np.log10(all_res_df['Enrichment_p'])
all_res_df.drop(columns=['Category'], inplace=True)

all_res_df.to_csv('key_results/SLDSC_results_from_sa_paper_both.csv', index=False)

#### For female

In [ ]:
all_res_df = None

for f in female_res_files:
    anno, pheno = f.split('iPSYCH-PGC_ASD_baseline_')[1].split('_female.results')[0].split('_sorted_control_')
    df = pd.read_csv(os.path.join(path, f), sep='\t')
    df = df[df['Category'] == 'L2_1']
    df['Annotation'] = anno
    df['Phenotype'] = pheno

    if all_res_df is None:
        all_res_df = df
    else:
        all_res_df = pd.concat([all_res_df, df], axis=0)

all_res_df['Group'] = all_res_df['Annotation'].apply(lambda x: assign_group(x))
all_res_df['neglog10p'] = -np.log10(all_res_df['Enrichment_p'])
all_res_df.drop(columns=['Category'], inplace=True)

all_res_df.to_csv('key_results/SLDSC_results_from_sa_paper_female.csv', index=False)

#### For male

In [ ]:
all_res_df = None

for f in male_res_files:
    anno, pheno = f.split('iPSYCH-PGC_ASD_baseline_')[1].split('_male.results')[0].split('_sorted_control_')
    df = pd.read_csv(os.path.join(path, f), sep='\t')
    df = df[df['Category'] == 'L2_1']
    df['Annotation'] = anno
    df['Phenotype'] = pheno

    if all_res_df is None:
        all_res_df = df
    else:
        all_res_df = pd.concat([all_res_df, df], axis=0)

all_res_df['Group'] = all_res_df['Annotation'].apply(lambda x: assign_group(x))
all_res_df['neglog10p'] = -np.log10(all_res_df['Enrichment_p'])
all_res_df.drop(columns=['Category'], inplace=True)

all_res_df.to_csv('key_results/SLDSC_results_from_sa_paper_male.csv', index=False)

In [ ]:
all_res_df

## for evolution annotations

In [ ]:
path = "UKB_Imaging_Genetics/LDSC_OUTPUT/SLDSC_OUTPUT/part_h2g_20230920_evolution"

In [ ]:
both_res_files = []
male_res_files = [] 
female_res_files = []

for f in os.listdir(path):
    if f.endswith('.results'):
        if "both" in f:
            both_res_files.append(f)
        elif "female" in f:
            female_res_files.append(f)
        elif "male" in f:
            male_res_files.append(f)

print("Both: ", len(both_res_files))
print("Female: ", len(female_res_files))
print("Male: ", len(male_res_files))

#### both

In [ ]:
all_res_df = None

for f in both_res_files:
    anno, pheno = f.split('iPSYCH-PGC_ASD_baseline_')[1].split('_both.results')[0].split('_sorted_control_')
    df = pd.read_csv(os.path.join(path, f), sep='\t')
    df = df[df['Category'] == 'L2_1']
    df['Annotation'] = anno
    df['Phenotype'] = pheno

    if all_res_df is None:
        all_res_df = df
    else:
        all_res_df = pd.concat([all_res_df, df], axis=0)

all_res_df['neglog10p'] = -np.log10(all_res_df['Enrichment_p'])
all_res_df.drop(columns=['Category'], inplace=True)

all_res_df.to_csv('key_results/SLDSC_results_evo_anno_both.csv', index=False)

#### female

In [ ]:
all_res_df = None

for f in female_res_files:
    anno, pheno = f.split('iPSYCH-PGC_ASD_baseline_')[1].split('_female.results')[0].split('_sorted_control_')
    df = pd.read_csv(os.path.join(path, f), sep='\t')
    df = df[df['Category'] == 'L2_1']
    df['Annotation'] = anno
    df['Phenotype'] = pheno

    if all_res_df is None:
        all_res_df = df
    else:
        all_res_df = pd.concat([all_res_df, df], axis=0)

all_res_df['neglog10p'] = -np.log10(all_res_df['Enrichment_p'])
all_res_df.drop(columns=['Category'], inplace=True)

all_res_df.to_csv('key_results/SLDSC_results_evo_anno_female.csv', index=False)

#### male

In [ ]:
all_res_df = None

for f in male_res_files:
    anno, pheno = f.split('iPSYCH-PGC_ASD_baseline_')[1].split('_male.results')[0].split('_sorted_control_')
    df = pd.read_csv(os.path.join(path, f), sep='\t')
    df = df[df['Category'] == 'L2_1']
    df['Annotation'] = anno
    df['Phenotype'] = pheno

    if all_res_df is None:
        all_res_df = df
    else:
        all_res_df = pd.concat([all_res_df, df], axis=0)

all_res_df['neglog10p'] = -np.log10(all_res_df['Enrichment_p'])
all_res_df.drop(columns=['Category'], inplace=True)

all_res_df.to_csv('key_results/SLDSC_results_evo_anno_male.csv', index=False)

In [ ]:
all_res_df['Annotation'].value_counts()

In [ ]:
all_res_df.sort_values(by=['Enrichment_p']).head(20)

---

In [ ]:
sns.scatterplot(data=all_res_df, x="Annotation", y="neglog10p", hue="Phenotype", palette="deep")
